# TranSTR — Grad-CAM Explainability (CausalVidQA)

Class-discriminative explanations for the predicted answer of `Train_full_mode` checkpoint.

**4 modalities visualized**:
1. Frame timeline Grad-CAM (target tensor: `frame_local`)
2. Object bbox Grad-CAM (target tensor: `obj_local`)
3. Question-token Grad-CAM (target tensor: `q_local`)
4. Unified-memory split Grad-CAM (target tensor: `mem`, split visual vs question)

Plus attention rollout (forward-only) for side-by-side comparison.

**Không cần train lại** — Grad-CAM hoàn toàn inference-time, dùng autograd trên checkpoint hiện có.

In [ ]:
# CELL 1: Git clone & cd
import os
REPO_URL = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH = 'full_mode'
if not os.path.exists(REPO_NAME):
    print(f'Cloning {REPO_URL}...')
    os.system(f'git clone {REPO_URL} -b {BRANCH}')
else:
    print('Repo already exists.')
if os.path.basename(os.getcwd()) != REPO_NAME:
    target_dir = os.path.join(os.getcwd(), REPO_NAME, 'causalvid')
    if os.path.exists(target_dir):
        os.chdir(target_dir)
    elif os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
print('Working dir:', os.getcwd())

In [ ]:
# CELL 2: Install + W&B (paste your key locally; left blank for GitHub safety)
import os
os.system('pip install -q wandb opencv-python einops --upgrade')
import wandb
WANDB_API_KEY = ''  # 🔴 Paste your W&B API key here before running
WANDB_PROJECT = 'transtr-causalvid-dino'
WANDB_ENTITY = None
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print('W&B logged in')
else:
    print('WANDB_API_KEY trống — bỏ qua tải artifact, set CHECKPOINT_PATH thủ công ở CELL 3.')

In [ ]:
# CELL 3: Download best checkpoint OR set local path
BEST_ARTIFACT_NAME = 'best-model-gdino-nolora-ncod-hum-run1'  # đổi tên tag tuỳ run
CHECKPOINT_PATH = ''  # nếu để trống và có WANDB_API_KEY thì sẽ tải từ W&B

if not CHECKPOINT_PATH and WANDB_API_KEY:
    api = wandb.Api()
    entity = WANDB_ENTITY or api.default_entity
    artifact_path = f'{entity}/{WANDB_PROJECT}/{BEST_ARTIFACT_NAME}:best'
    print('Downloading', artifact_path)
    art = api.artifact(artifact_path)
    artifact_dir = art.download()
    # accept either old or new naming
    cands = [f for f in os.listdir(artifact_dir) if f.endswith('.ckpt')]
    CHECKPOINT_PATH = os.path.join(artifact_dir, cands[0]) if cands else ''

assert CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH), (
    f'CHECKPOINT_PATH không tồn tại: {CHECKPOINT_PATH}. Hãy điền tay ở CELL này.'
)
print('Checkpoint:', CHECKPOINT_PATH)

In [ ]:
# CELL 4: Imports + plot styles
import json
import pickle as pkl
from itertools import chain

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from einops import rearrange
from IPython.display import Video as IPyVideo, display, HTML

from utils.util import set_seed, set_gpu_devices
from networks.model import VideoQAmodel
from networks.topk import PerturbedTopK, HardtopK  # noqa: F401

# Grad-CAM utilities
from explain.gradcam_hooks import MultiTargetGradCAM
from explain.viz_gradcam import (
    plot_frame_gradcam,
    plot_object_gradcam,
    plot_question_token_gradcam,
    plot_mem_split_bar,
    plot_attention_vs_gradcam,
)

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'font.size': 11,
})
print('Imports OK')

In [ ]:
# CELL 5: Configuration — auto-compatible with checkpoint obj feature dim (1028 or 1796)
CLIP_SPLIT_PATH = '/kaggle/input/datasets/danielq07/dinov3-feat/features'
CLIP_MERGED_PATH = '/kaggle/working/dinov3_T16_dim1024_merge'
AUTO_DETECT_OBJ_DIM = True

class DebugConfig:
    video_feature_root = CLIP_MERGED_PATH
    grounding_dino_path = '/kaggle/input/datasets/danielq07/GroundingDINO-full-feature'
    sample_list_path = '/kaggle/input/datasets/danielq07/casualqa-test/QA_test'
    split_dir_txt = '/kaggle/input/casual-vid-data-split/split'
    topK_frame = 16
    frames = 16
    select_frames = 5
    objs = 20
    obj_topK = 12
    frame_feat_dim = 1024
    obj_feat_dim = 1796   # default, will auto-override from checkpoint when AUTO_DETECT_OBJ_DIM=True
    use_grounding_dino = True
    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 1e-5
    text_pool_mode = 1
    use_lora = False
    n_query = 5
    return_family_id = True
    hard_eval = False

args = DebugConfig()
set_gpu_devices(0)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 🔁 Đổi VIDEO_ID/folder cho từng case
VIDEO_FOLDER = '/kaggle/input/datasets/danielq07/raw-casual/root/Desktop/Data/test'
VIDEO_ID = '9OE8Krqn3bo_000078_000088'
VIDEO_MP4_PATH = os.path.join(VIDEO_FOLDER, f'{VIDEO_ID}.mp4')

QUESTION_TYPES = ['descriptive', 'explanatory', 'predictive', 'predictive_reason',
                  'counterfactual', 'counterfactual_reason']
QTYPE_DISPLAY = {
    'descriptive': 'Descriptive (D)',
    'explanatory': 'Explanatory (E)',
    'predictive': 'Predictive Answer (PA)',
    'predictive_reason': 'Predictive Reason (PR)',
    'counterfactual': 'Counterfactual Answer (CA)',
    'counterfactual_reason': 'Counterfactual Reason (CR)',
}
FAMILY_TO_ID = {k: i for i, k in enumerate(QUESTION_TYPES)}

print('Device:', device, '| Video:', VIDEO_ID)
if not os.path.exists(VIDEO_MP4_PATH):
    print(f'❌ Không thấy video: {VIDEO_MP4_PATH}')
    if os.path.exists(VIDEO_FOLDER):
        print('Folder list (5 đầu):', os.listdir(VIDEO_FOLDER)[:5])
    raise FileNotFoundError(VIDEO_MP4_PATH)
print('✅ Video tồn tại:', VIDEO_MP4_PATH)

In [ ]:
# CELL 6: Đảm bảo có DINOv3 .pt cho VIDEO_ID (copy từ split source nếu cần)
import shutil
from tqdm.auto import tqdm

target_pt = os.path.join(CLIP_MERGED_PATH, f'{VIDEO_ID}.pt')
os.makedirs(CLIP_MERGED_PATH, exist_ok=True)
if os.path.exists(target_pt):
    print('✅ Feature exists:', target_pt)
else:
    found = False
    for split in ['train', 'test', 'valid', 'val']:
        cand = os.path.join(CLIP_SPLIT_PATH, split, f'{VIDEO_ID}.pt')
        if os.path.exists(cand):
            shutil.copy2(cand, target_pt)
            print(f'  copied {split}/{VIDEO_ID}.pt')
            found = True
            break
    assert os.path.exists(target_pt), f'Không có {target_pt}, kiểm tra CLIP_SPLIT_PATH/VIDEO_ID.'

In [ ]:
# CELL 7: Build model + load checkpoint (auto-detect obj_feat_dim from checkpoint)
state_raw = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = state_raw['model_state_dict'] if isinstance(state_raw, dict) and 'model_state_dict' in state_raw else state_raw

if AUTO_DETECT_OBJ_DIM and 'obj_resize.fc.weight' in state_dict:
    ckpt_obj_dim = int(state_dict['obj_resize.fc.weight'].shape[1])
    args.obj_feat_dim = ckpt_obj_dim
    print(f'🔎 Auto-detected obj_feat_dim from checkpoint: {ckpt_obj_dim}')
else:
    print(f'⚠️ Could not auto-detect obj_feat_dim, keep config value: {args.obj_feat_dim}')

cfg = {k: v for k, v in DebugConfig.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames  # 5 frames internally
cfg['obj_feat_dim'] = args.obj_feat_dim

model = VideoQAmodel(**cfg).to(device)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
model.eval()
print(f'Loaded. missing={len(missing)} unexpected={len(unexpected)}')
print(f'Model obj_feat_dim={args.obj_feat_dim}')

In [ ]:
# CELL 8: Asset helpers (decode MP4, load DINOv3 + GroundingDINO features; supports obj dim 1028/1796)
OBJ_DIM = args.obj_feat_dim


def extract_video_frames(video_path, n_frames=16):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video {video_path}')
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or n_frames
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    indices = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            frame = np.zeros((360, 640, 3), dtype=np.uint8)
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, indices.tolist(), total, fps


def load_dinov3_features(video_id):
    path = os.path.join(args.video_feature_root, f'{video_id}.pt')
    feat = torch.load(path, map_location='cpu').float()
    nf = feat.shape[0]
    if nf > args.topK_frame:
        idx = np.linspace(0, nf - 1, args.topK_frame).astype(int)
        feat = feat[idx]
    elif nf < args.topK_frame:
        feat = torch.cat([feat, torch.zeros(args.topK_frame - nf, feat.shape[1])], 0)
    return feat


def _normalize_boxes(boxes, orig_w, orig_h):
    bb = boxes.astype(np.float32)
    if bb.size and bb.max() > 1:
        bb = bb / np.array([orig_w, orig_h, orig_w, orig_h], dtype=np.float32)
    return bb


def _build_obj_row(roi, cls_emb, boxes, orig_w, orig_h):
    """Build per-box feature according to OBJ_DIM.

    - 1028: [roi(1024) | bbox(4)]
    - 1796: [roi(1024) | cls_emb(768) | bbox(4)]
    - else: pad/truncate from richest representation [roi|cls_emb|bbox]
    """
    n = len(roi)
    if n == 0:
        return np.zeros((0, OBJ_DIM), dtype=np.float32)

    roi = roi.astype(np.float32)
    bb = _normalize_boxes(boxes, orig_w, orig_h)

    if OBJ_DIM == 1028:
        base = np.concatenate([roi, bb], axis=-1)
        return base.astype(np.float32)

    if cls_emb is None or len(cls_emb) == 0:
        cls_emb = np.zeros((n, 768), dtype=np.float32)
    else:
        cls_emb = cls_emb.astype(np.float32)

    rich = np.concatenate([roi, cls_emb, bb], axis=-1)  # 1796

    if OBJ_DIM == 1796:
        return rich.astype(np.float32)

    if rich.shape[1] > OBJ_DIM:
        return rich[:, :OBJ_DIM].astype(np.float32)

    pad = np.zeros((n, OBJ_DIM - rich.shape[1]), dtype=np.float32)
    return np.concatenate([rich, pad], axis=-1).astype(np.float32)


def load_grounding_dino(video_id):
    path = os.path.join(args.grounding_dino_path, f'{video_id}.pkl')
    with open(path, 'rb') as f:
        data = pkl.load(f)

    frames_data = data.get('frames', [])
    orig_h = data.get('orig_h', 1080)
    orig_w = data.get('orig_w', 1920)
    if not frames_data:
        raise RuntimeError(f'No GDINO frames for {video_id}')

    total = len(frames_data)
    sample_idx = (np.linspace(0, total - 1, args.topK_frame).astype(int)
                  if total >= args.topK_frame else np.arange(total))

    objs = []
    for i in sample_idx:
        fd = frames_data[int(i)]
        roi = np.asarray(fd.get('roi_features', np.zeros((0, 1024), dtype=np.float32)))
        cls_emb = fd.get('class_text_embedding', None)
        cls_emb = np.asarray(cls_emb) if cls_emb is not None else None
        boxes = np.asarray(fd.get('boxes_xyxy_orig', np.zeros((0, 4), dtype=np.float32)))

        mat = _build_obj_row(roi, cls_emb, boxes, orig_w, orig_h)
        mat = torch.from_numpy(mat).float()

        if mat.shape[0] > args.objs:
            mat = mat[:args.objs]
        elif mat.shape[0] < args.objs:
            mat = torch.cat([mat, torch.zeros(args.objs - mat.shape[0], OBJ_DIM)], 0)

        objs.append(mat)

    while len(objs) < args.topK_frame:
        objs.append(torch.zeros(args.objs, OBJ_DIM))

    return torch.stack(objs), frames_data, sample_idx, orig_h, orig_w


def prepare_video_assets(video_id, video_path):
    frames, indices, total, fps = extract_video_frames(video_path, args.topK_frame)
    ff = load_dinov3_features(video_id)
    of_tensor, gd_frames, gd_sample_idx, oh, ow = load_grounding_dino(video_id)
    return {
        'frames': frames,
        'frame_indices': indices,
        'total_frames': total,
        'fps': fps,
        'frame_feat': ff,
        'obj_feat': of_tensor,
        'gdino_frames': gd_frames,
        'gdino_sample_idx': gd_sample_idx,
        'orig_h': oh,
        'orig_w': ow,
    }


def load_question_bundle(video_id, qtype):
    base = qtype.replace('_reason', '')
    with open(os.path.join(args.sample_list_path, video_id, 'text.json'), encoding='utf-8') as f:
        text_data = json.load(f)
    with open(os.path.join(args.sample_list_path, video_id, 'answer.json'), encoding='utf-8') as f:
        ans_data = json.load(f)

    block = text_data[base]
    question = block['question']
    if 'reason' in qtype:
        options = block['reason']
        correct_idx = ans_data[base]['reason']
    else:
        options = block['answer']
        correct_idx = ans_data[base]['answer']

    qns_word = (question,)
    ans_word = [tuple(f'{question} [SEP] {opt}' for opt in options)]
    return {
        'question': question,
        'answers': options,
        'correct_idx': correct_idx,
        'qns_word': qns_word,
        'ans_word': ans_word,
        'family_id': FAMILY_TO_ID.get(qtype, 0),
    }


print(f'Asset helpers ready (OBJ_DIM={OBJ_DIM})')

In [ ]:
# CELL 9: Single-question runner — Grad-CAM + visualization
from explain.gradcam_hooks import MultiTargetGradCAM

# Reuse one explainer across cells (no model edit, hooks are inside `_forward_hooked`)
explainer = MultiTargetGradCAM(model)

def run_question_explain(video_assets, qtype, per_class=False):
    qa = load_question_bundle(VIDEO_ID, qtype)
    print(f'\n{"=" * 80}\n{QTYPE_DISPLAY[qtype]}\n{"=" * 80}')
    print('Q:', qa['question'])
    for i, opt in enumerate(qa['answers']):
        flag = '✓' if i == qa['correct_idx'] else ' '
        print(f'  [{i}] {flag} {opt}')

    ff = video_assets['frame_feat'].unsqueeze(0).to(device)
    of = video_assets['obj_feat'].unsqueeze(0).to(device)
    fam = torch.tensor([qa['family_id']], device=device)

    out = explainer.run(
        ff, of, qa['qns_word'], qa['ans_word'],
        device=device, q_family_id=fam, per_class=per_class,
    )
    pred = out['pred']
    verdict = '✅ correct' if pred == qa['correct_idx'] else '❌ wrong'
    print(f"Predicted: [{pred}] {qa['answers'][pred]}  ({verdict}, P={out['probs'][pred]:.3f})")

    cams = out['cams']
    # Frame attention peak for compare
    frame_att_peak = out['frame_att'].max(axis=-1)  # [F_total]

    # 1) Frame Grad-CAM
    plot_frame_gradcam(
        video_assets['frames'], cams['frame_cam'][0],
        out['selected_frame_indices'], out['rollout_frame'],
        qa['question'], QTYPE_DISPLAY[qtype],
    )
    # 2) Object Grad-CAM
    if cams['obj_cam'] is not None:
        plot_object_gradcam(
            video_assets['frames'], out['selected_frame_indices'],
            cams['obj_cam'][0], video_assets['gdino_frames'],
            video_assets['gdino_sample_idx'], video_assets['orig_h'], video_assets['orig_w'],
            QTYPE_DISPLAY[qtype],
        )
    # 3) Question token Grad-CAM
    if cams['q_cam'] is not None:
        plot_question_token_gradcam(
            model.tokenizer, qa['question'], cams['q_cam'][0],
            QTYPE_DISPLAY[qtype], answers=qa['answers'], pred_idx=pred,
        )
    # 4) Mem split
    if cams['mem_visual_cam'] is not None and cams['mem_question_cam'] is not None:
        plot_mem_split_bar(
            cams['mem_visual_cam'][0], cams['mem_question_cam'][0],
            out['selected_frame_indices'], QTYPE_DISPLAY[qtype],
        )
    # 5) Compare attention vs gradcam vs rollout
    plot_attention_vs_gradcam(
        frame_att_peak, cams['frame_cam'][0], out['rollout_frame'],
        out['selected_frame_indices'], QTYPE_DISPLAY[qtype],
    )

    return {'qtype': qtype, 'pred': pred, 'correct': qa['correct_idx'],
            'conf': float(out['probs'][pred]), 'out': out, 'qa': qa}

print('run_question_explain ready')

In [ ]:
# CELL 10: Preview video
if os.path.exists(VIDEO_MP4_PATH):
    display(HTML(f'<b>Video ID:</b> {VIDEO_ID}<br><b>Folder:</b> {VIDEO_FOLDER}'))
    display(IPyVideo(VIDEO_MP4_PATH, embed=True, width=720))

In [ ]:
# CELL 11: Prepare features
video_assets = prepare_video_assets(VIDEO_ID, VIDEO_MP4_PATH)
print(f"Frames: {len(video_assets['frames'])}/{video_assets['total_frames']} fps={video_assets['fps']:.1f}")
print('DINOv3 feat:', tuple(video_assets['frame_feat'].shape),
      '| GDINO obj:', tuple(video_assets['obj_feat'].shape))

In [ ]:
# CELL 12: Descriptive (D)
_ = run_question_explain(video_assets, 'descriptive')

In [ ]:
# CELL 13: Explanatory (E)
_ = run_question_explain(video_assets, 'explanatory')

In [ ]:
# CELL 14: Predictive Answer (PA)
_ = run_question_explain(video_assets, 'predictive')

In [ ]:
# CELL 15: Predictive Reason (PR)
_ = run_question_explain(video_assets, 'predictive_reason')

In [ ]:
# CELL 16: Counterfactual Answer (CA)
_ = run_question_explain(video_assets, 'counterfactual')

In [ ]:
# CELL 17: Counterfactual Reason (CR)
_ = run_question_explain(video_assets, 'counterfactual_reason')

## (Optional) Counterfactual class explanation

Chạy Grad-CAM riêng cho từng class candidate để biết "nếu model chọn A_j thì nó dựa vào đâu". Dùng `per_class=True` trong `run_question_explain` rồi truy cập `result['out']['per_class_cams'][j]`.